In [ ]:
# pip install rembg[gpu] onnxruntime-gpu pillow numpy torch
# git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2

import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from rembg import new_session, remove

sys.path.insert(0, "Hunyuan3D-2")
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

RAW_DIR = Path("input/raw_images")
BGR_DIR = Path("outputs/bgr_images")
STL_DIR = Path("outputs/hunyuan_stls")

MODEL_PATH = "tencent/Hunyuan3D-2.1"
MODEL_SUBFOLDER = "hunyuan3d-dit-v2-1"
SEED = 1234
NUM_INFERENCE_STEPS = 30
OCTREE_RESOLUTION = 300
GUIDANCE_SCALE = 7.5
DUAL_GUIDANCE_SCALE = 10.5
WHITE_THRESHOLD = 250
BATCH_SIZE = 16
DEVICE = "cuda:0"


def remove_background(sess, src, dst):
    im = Image.open(src).convert("RGB")
    rgba = remove(im, session=sess)
    white = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
    white.paste(rgba, (0, 0), rgba)
    dst.parent.mkdir(parents=True, exist_ok=True)
    white.convert("RGB").save(dst, "PNG")


def load_as_rgba_white_bg(path):
    rgb = np.array(Image.open(path).convert("RGB"))
    fg = np.any(rgb < WHITE_THRESHOLD, axis=-1)
    alpha = np.where(fg, 255, 0).astype(np.uint8)
    return Image.fromarray(np.dstack([rgb, alpha]), mode="RGBA")


def save_mesh(mesh, out):
    out.parent.mkdir(parents=True, exist_ok=True)
    tmp = out.with_name(f".{out.name}.tmp")
    mesh.export(str(tmp), file_type="stl")
    os.replace(str(tmp), str(out))


def reconstruct(pipeline, images):
    return pipeline(
        image=images,
        num_inference_steps=NUM_INFERENCE_STEPS,
        octree_resolution=OCTREE_RESOLUTION,
        guidance_scale=GUIDANCE_SCALE,
        dual_guidance_scale=DUAL_GUIDANCE_SCALE,
        generator=torch.Generator(device=pipeline.device).manual_seed(SEED),
    )


sess = new_session("birefnet-general", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
raw = sorted(p for p in RAW_DIR.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
t0 = time.time()
for i, src in enumerate(raw, 1):
    dst = BGR_DIR / src.relative_to(RAW_DIR).with_suffix(".png")
    if not dst.exists():
        remove_background(sess, src, dst)
    if i % 100 == 0 or i == len(raw):
        print(f"background removal {i}/{len(raw)}  {(time.time() - t0) / 60:.1f} min", flush=True)
del sess

pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    MODEL_PATH, subfolder=MODEL_SUBFOLDER, use_safetensors=False, device=DEVICE
)
jobs = [(p, STL_DIR / p.relative_to(BGR_DIR).with_suffix(".stl")) for p in sorted(BGR_DIR.rglob("*.png"))]
jobs = [(p, o) for p, o in jobs if not o.exists()]
t0 = time.time()
for s in range(0, len(jobs), BATCH_SIZE):
    batch = jobs[s:s + BATCH_SIZE]
    try:
        meshes = reconstruct(pipeline, [load_as_rgba_white_bg(p) for p, _ in batch])
    except Exception as e:
        print(f"batch failed ({e}), retrying one by one", flush=True)
        torch.cuda.empty_cache()
        meshes = []
        for p, _ in batch:
            try:
                meshes.append(reconstruct(pipeline, load_as_rgba_white_bg(p))[0])
            except Exception as e1:
                print(f"failed {p.name}: {e1}", flush=True)
                meshes.append(None)
    for (p, out), mesh in zip(batch, meshes):
        if mesh is None:
            print(f"empty mesh {p.name}", flush=True)
            continue
        save_mesh(mesh, out)
    done = s + len(batch)
    print(f"reconstruction {done}/{len(jobs)}  {(time.time() - t0) / 60:.1f} min", flush=True)